# Support Vector Machines: Exercises

This notebook accompanies the `10_svm` module. You will train the from-scratch
`LinearSVM` and `KernelSVM` implementations on synthetic outlet data, then
explore how `C` and `gamma` change the decision boundary.

## Learning objectives

- Fit a linear SVM with Pegasos and inspect its objective.
- See how L2 regularization (`lambda_`) controls the weight norm.
- Compare linear, RBF and polynomial kernels.
- Tune `C` and `gamma` with scikit-learn's `SVC`.


In [ ]:
import os
import sys

import numpy as np
import pandas as pd

# Make the module importable when the notebook runs from this directory.
sys.path.insert(0, os.path.abspath("."))

from implementation import (
    OUTLET_FEATURE_NAMES,
    KernelSVM,
    LinearSVM,
    generate_outlet_data,
)

np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Explore the synthetic outlet data

`generate_outlet_data` creates a reproducible dataset. The label is based on
sales per staff member compared with a traffic-dependent benchmark, so the
class boundary is non-linear in the raw features.


In [ ]:
X, y = generate_outlet_data(n_samples=400, random_state=RANDOM_STATE)

df = pd.DataFrame(X, columns=OUTLET_FEATURE_NAMES)
df["high_performing"] = y

print(df.describe().round(2))
print("\nClass balance:")
print(df["high_performing"].value_counts(normalize=True).round(3))
df.head()


## 2. Split and standardize

SVMs are distance- and margin-based, so feature scaling matters. Our estimators
standardize internally by default; here we also prepare scaled arrays for the
scikit-learn comparison later.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 3. Linear SVM with Pegasos

Fit `LinearSVM`, then inspect training/test accuracy, the weight norm and the
objective history. The full objective is the regularized hinge loss, so it
should trend downward even though Pegasos is stochastic.


In [ ]:
linear = LinearSVM(lambda_=1e-3, n_epochs=60, random_state=RANDOM_STATE)
linear.fit(X_train, y_train)

print(f"Train accuracy: {linear.score(X_train, y_train):.4f}")
print(f"Test accuracy:  {linear.score(X_test, y_test):.4f}")
print(f"||w||:          {np.linalg.norm(linear.w):.4f}")
print(f"b:              {linear.b:.4f}")
print(f"First/last objective: {linear.train_losses_[0]:.4f} -> {linear.train_losses_[-1]:.4f}")


## 4. Exercise: the effect of `lambda_`

Train linear SVMs with `lambda_` values `1e-4`, `1e-3` and `1e-2`. Record the
weight norm and test accuracy. Larger `lambda_` means stronger L2
regularization.

**Question:** Does the weight norm shrink as `lambda_` grows? Does test accuracy
change monotonically?


In [ ]:
results = []
for lam in [1e-4, 1e-3, 1e-2]:
    model = LinearSVM(lambda_=lam, n_epochs=60, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    results.append((lam, np.linalg.norm(model.w), model.score(X_test, y_test)))

print(f"{'lambda':>10} {'||w||':>10} {'test acc':>10}")
for lam, norm, acc in results:
    print(f"{lam:>10.0e} {norm:>10.3f} {acc:>10.4f}")


## 5. Exercise: compare kernels

Train `KernelSVM` with linear, RBF and polynomial kernels at `C=10`. Compare
accuracy, the number of support vectors and the number of SMO passes.

**Question:** Which kernel fits the non-linear traffic benchmark best? Does the
most flexible kernel always win on the test set?


In [ ]:
for kernel in ["linear", "rbf", "poly"]:
    model = KernelSVM(
        C=10.0,
        kernel=kernel,
        gamma="scale",
        max_iter=100,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    print(
        f"{kernel:>6}: test acc={model.score(X_test, y_test):.4f}, "
        f"support vectors={model.n_support_:>3}, passes={model.n_iter_:>3}"
    )


## 6. Exercise: tune `C` and `gamma` with scikit-learn

The grid below mirrors `sklearn_example.py`. Use a coarse logarithmic grid,
then inspect the best test configuration. In a real project you would select
hyperparameters with cross-validation on the training set only, not on the test
set.


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

rows = []
for C in [0.1, 1.0, 10.0, 100.0]:
    for gamma in [0.01, 0.1, 0.5, 1.0]:
        svc = SVC(kernel="rbf", C=C, gamma=gamma)
        svc.fit(X_train_scaled, y_train)
        train_acc = accuracy_score(y_train, svc.predict(X_train_scaled))
        test_acc = accuracy_score(y_test, svc.predict(X_test_scaled))
        rows.append((C, gamma, train_acc, test_acc))

table = pd.DataFrame(rows, columns=["C", "gamma", "train_acc", "test_acc"])
print(table.sort_values("test_acc", ascending=False).head(8).to_string(index=False))


## 7. Challenge: out-of-sample diagnosis

1. Split the outlet data into training, validation and test sets.
2. Tune `C` and `gamma` using only the validation set.
3. Report the final test accuracy once.
4. Compare the tuned RBF SVM with a tuned linear SVM.

**Discussion:** Why can a high training accuracy coexist with a low test
accuracy? Relate your answer to `C`, `gamma` and the number of support vectors.


## 8. Challenge: implement a new kernel

Add a `sigmoid` kernel to `KernelSVM`:

```
K(x, z) = tanh(gamma * x · z + coef0)
```

Update `_kernel_matrix` and the kernel validation, then compare it with the
three existing kernels. Explain why the kernel matrix must be positive
semi-definite for the dual problem to remain well-behaved.
